# C&W L2 — Offline Residual CNN robustness on **your road-sign photos**

Uses four crops from your five uploaded photos. Classes: **35 mph** (two samples), **45 mph** (two samples). `IMG_2686.jpeg` is retained but excluded because its sign is cut off. The trained Keras model is required. No camera stream or physical sign changes are involved.

**Important:** results are *calculated* here after you upload trained weights. The notebook does not pre-fill success percentages.

In [ ]:
%pip -q install tensorflow numpy matplotlib pillow

## 1. Upload three files

When the upload dialog appears, select these three files together:

1. `cw_l2_photo_simulation.zip` (the supplied ZIP containing the photos, script, and this notebook)
2. `residual_cnn_traffic_signs.keras` (trained with the earlier Residual CNN training notebook)
3. `residual_classes.json` (downloaded with those model weights)

The upload is only to your Colab runtime.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded=files.upload()
required={'cw_l2_photo_simulation.zip','residual_cnn_traffic_signs.keras','residual_classes.json'}
missing=required-set(uploaded)
if missing:
    raise FileNotFoundError(f'Missing uploaded input files: {sorted(missing)}')

with zipfile.ZipFile('cw_l2_photo_simulation.zip') as archive:
    archive.extractall('/content')
BUNDLE=Path('/content/cw_l2_photo_simulation')
assert (BUNDLE/'cw_l2_offline.py').is_file(), 'Could not find script in ZIP'
print('Images:', len(list((BUNDLE/'sign_crops').glob('*.jpg'))))

## 2. Review the four real sign crops

The blurred 45 mph image is kept to show the difference between poor clean recognition and *actual* adversarial errors.

In [ ]:
import matplotlib.pyplot as plt
import json
from PIL import Image

manifest=json.loads((BUNDLE/'photo_manifest.json').read_text())
fig, axes=plt.subplots(1,len(manifest),figsize=(12,3))
for ax,info in zip(axes,manifest):
    ax.imshow(Image.open(BUNDLE/info['image']))
    ax.set_title(f"{info['source']}\n{info['true_label']}")
    ax.axis('off')
plt.tight_layout();plt.show()

## 3. Run offline Carlini–Wagner L2

Carlini–Wagner minimizes L2 pixel change plus a classification margin. It only optimizes images that the model originally classifies correctly. If a sign is already wrong due to blur/lighting, it is **not counted as an attack success**.

`steps` and `binary-steps` are experimental iteration counts, not accuracy targets.

In [ ]:
import subprocess,sys
cmd=[
    sys.executable,str(BUNDLE/'cw_l2_offline.py'),
    '--model','/content/residual_cnn_traffic_signs.keras',
    '--classes','/content/residual_classes.json',
    '--output',str(BUNDLE/'cw_results'),
    '--steps','160',
    '--binary-steps','4',
]
subprocess.run(cmd,check=True)

## 4. Read the measured results

In [ ]:
import json
import pandas as pd
from IPython.display import display,Image as NotebookImage

results_dir=BUNDLE/'cw_results'
summary=json.loads((results_dir/'cw_summary.json').read_text())
print(json.dumps(summary,indent=2))
df=pd.read_csv(results_dir/'cw_individual_results.csv')
display(df[['source','true_label','clean_prediction','attacked_prediction',
            'clean_true_class_confidence','attacked_true_class_confidence',
            'true_class_confidence_change','cw_attack_success','l2_distortion']])
display(NotebookImage(filename=str(results_dir/'cw_comparison.png')))
display(NotebookImage(filename=str(results_dir/'cw_accuracy_chart.png')))

## 5. Download the simulation results

The results ZIP contains the CSV, JSON, before/after figures, and saved offline test crops. The originally uploaded photos remain untouched.

In [ ]:
import zipfile
result_zip=Path('/content/cw_measured_results.zip')
with zipfile.ZipFile(result_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in results_dir.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(results_dir))
files.download(str(result_zip))